import the libraries important for data cleaning and preparation

In [1]:
import pandas as pd
import camelot
import warnings

some tweaks to supress irrelevant warnings and optimize some settings

In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns',  None)

Convert the pdf into csv

In [3]:
tables = camelot.read_pdf('../data/auction3.pdf', pages='all', flavor='lattice')
combined = pd.concat([table.df for table in tables], ignore_index=True)
combined.to_csv('../data/clean_auction3.csv')

import the csv file for manipulation and cleaning give the dataset the correct columns name

In [4]:
df = pd.read_csv('../data/clean_auction3.csv')
df.columns = ['one', 'no', 'rank', 'winners', 'price_per_sqm', 'down_payment_pct', 'subcity', 'district', 'sqm', 'code']
print(f"The shape of the first dataset is {df.shape}")

The shape of the first dataset is (1567, 10)


In [5]:
pd.set_option('display.max_rows', 2000)
df.head(1200)

,one,no,rank,winners,price_per_sqm,down_payment_pct,subcity,district,sqm,code
0,0,በቂርቆስ ክ/ክተማ የአሸናፊዎች ዝርዝር,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,ተ.ቁ,ደረጃ,የአሸናፊዎች ስም ዝርዝር,ለካሬ ሜትር የተ,ድመ ክፍ\nያ በ% \nሰጠ ዋጋ ቅ,ክ/ከተማ,ወረዳ,የቦታው ስፋ\nት በካ.ሜ,የቦታ ኮድ
2,2,1,1,ጀማሉዲን ኤሊያስ መሐመድ,"48,205.00",100%,ቂርቆስ,08,963,LDR-KIR-MIX-00014228
3,3,NaN,2,ሃብቶም ገብሬ ጋይም,"31,321.00",100%,NaN,NaN,NaN,NaN
4,4,NaN,3,ሳዲቅ ሲራጅ አደም,"42,177.00",56%,NaN,NaN,NaN,NaN
5,5,2,1,ፍቅሩ ገነቲ ሁንዴ,"94,200.00",100%,ቂርቆስ,08,1094,LDR-KIR-MIX-00014229
6,6,NaN,2,ያሬድ ግርማ በቀለ,"77,110.00",100%,NaN,NaN,NaN,NaN
7,7,NaN,3,ዩኒቲ ሪልስቴት አክስዮን ማኀበር,"101,355.00",50%,NaN,NaN,NaN,NaN
8,8,3,1,አስፋው አጆላ ሀይ,"86,700.00",100%,ቂርቆስ,08,1274,LDR-KIR-MIX-00014230
9,9,NaN,2,ሀርዴ ትሬዲንግ ኃ/የተ/የግል ማኀበር,"58,500.00",100%,NaN,NaN,NaN,NaN


drop irrelevant columns

In [6]:
df.drop(columns=['one', 'no', 'rank', 'winners', 'code'], inplace=True)

drop irrelevant and empty rows, then rearrange the index

In [7]:
df = df.dropna(subset=['price_per_sqm'])
df = df[df['price_per_sqm'].astype(str).str.contains(r'\d', regex=True, na=False)]
df = df.reset_index(drop=True)

rename the subcity names into the appropriate kinda names

In [8]:
df.loc[0:23, 'subcity'] = 'Kirkos'
df.loc[24:83, 'subcity'] = 'Kolfe Keranyo'
df.loc[84:92, 'subcity'] = 'Gulele'
df.loc[93:134, 'subcity'] = 'Yeka'
df.loc[135:260, 'subcity'] = 'Lemi Kura'
df.loc[261:701, 'subcity'] = 'Akaki Kality'
df.loc[702:758, 'subcity'] = 'Addis Ketema'
df.loc[759:, 'subcity'] = 'Nefas - Silk Lafto'

inspect the data types and change them to the appropriate ones

In [9]:
df['district'] = df['district'].ffill(limit=2)
df['price_per_sqm'] = df['price_per_sqm'].astype(str).str.split().str[0]  # Keep "12,500.00"
df['down_payment_pct'] = df['down_payment_pct'].fillna(100)
df['sqm'] = df['sqm'].ffill(limit=2)
df['price_per_sqm'] = df['price_per_sqm'].astype(str).str.replace(',', '').str.strip().astype('float64')
df['down_payment_pct'] = df['down_payment_pct'].astype(str).str.replace('%', '').str.strip().astype('float64')
df['district'] = df['district'].astype(str).str.replace('ዏ', '0')
df['sqm'] = df['sqm'].astype(str).str.replace('ዏ', '0').astype('float64')
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1139 entries, 0 to 1138
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   price_per_sqm     1139 non-null   float64
 1   down_payment_pct  1139 non-null   float64
 2   subcity           1139 non-null   str    
 3   district          800 non-null    str    
 4   sqm               1130 non-null   float64
dtypes: float64(3), str(2)
memory usage: 44.6 KB


None

In [10]:
df.to_csv('../data/clean_auction3.csv', index=False)